# NB 02 — Distributed Arabic Preprocessing

**Phase 2 — Tasks 2.1 / 2.2**

In this notebook:
- Loads the labelled raw DataFrame
- Applies the Arabic NLP preprocessing pipeline (Spark UDFs):
  normalisation, diacritic removal, stop-word removal, light stemming
- Saves the preprocessed DataFrame as Parquet (Task 2.2)
- Demonstrates MapReduce-style corpus statistics (Task 2.1)


In [ ]:
# ── Session bootstrap ─────────
from google.colab import drive
drive.mount('/content/drive')

import os, sys, shutil, importlib
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/MSBDA-801-Project/arabic_ai_detection')
SRC_DIR = PROJECT_ROOT / 'src'

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f'Project root not found: {PROJECT_ROOT}')

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# src/__init__.py is committed to the repo, no runtime touch needed
print('PROJECT_ROOT:', PROJECT_ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_ROOT: /content/drive/MyDrive/MSBDA-801-Project/arabic_ai_detection


In [ ]:
import importlib
import src.data_preparation as data_prep
importlib.reload(data_prep)

from src.utils import (
    create_spark_session, load_checkpoint, save_checkpoint,
    checkpoint_exists, add_src_to_spark, Timer, FIGURES_DIR
)
from src.data_preparation import (
    download_dataset, build_labelled_dataframe,
    apply_preprocessing, mapreduce_word_count, mapreduce_ngram_frequency,
)
from pyspark.sql import functions as F
import pandas as pd
import matplotlib.pyplot as plt

spark = create_spark_session('ArabicAIDetection_Preprocessing')
add_src_to_spark(spark)


'/content/drive/MyDrive/MSBDA-801-Project/arabic_ai_detection/src_package.zip'

## 1. Load (or create) labelled raw dataset

In [ ]:
import shutil

raw_files = list(__import__('src.utils', fromlist=['DATA_RAW']).DATA_RAW.glob('*.parquet'))
if len(raw_files) < 3:
    print('Raw files not found. Downloading …')
    download_dataset()

if checkpoint_exists('labelled_raw'):
    labelled_df = load_checkpoint(spark, 'labelled_raw')
    print('Loaded labelled_raw checkpoint.')
else:
    labelled_df = build_labelled_dataframe(spark)
    save_checkpoint(labelled_df, 'labelled_raw')

labelled_df = labelled_df.cache()
print(f'{labelled_df.count():,} rows loaded.')
labelled_df.printSchema()


Loaded labelled_raw checkpoint.
41,940 rows loaded.
root
 |-- text: string (nullable = true)
 |-- label: integer (nullable = true)
 |-- source_model: string (nullable = true)
 |-- generation_method: string (nullable = true)



## 2. Apply Arabic preprocessing


In [ ]:
FORCE_REBUILD_PREPROCESSED = False   # ← set True after changing data_preparation.py

from src.utils import DATA_PROCESSED

if FORCE_REBUILD_PREPROCESSED:
    import shutil
    p = DATA_PROCESSED / 'preprocessed'
    if p.exists():
        shutil.rmtree(p)
        print('Deleted old preprocessed checkpoint.')

if checkpoint_exists('preprocessed'):
    preprocessed_df = load_checkpoint(spark, 'preprocessed')
    print('Loaded preprocessed checkpoint.')
else:
    with Timer('Arabic preprocessing'):
        preprocessed_df = apply_preprocessing(labelled_df)
        save_checkpoint(preprocessed_df, 'preprocessed')
    print('Preprocessing complete and checkpoint saved.')

preprocessed_df = preprocessed_df.cache()
preprocessed_df.printSchema()


Preprocessing complete and checkpoint saved.
root
 |-- text: string (nullable = true)
 |-- label: integer (nullable = true)
 |-- source_model: string (nullable = true)
 |-- generation_method: string (nullable = true)
 |-- clean_text: string (nullable = true)



## 3. Before / after preprocessing examples

In [ ]:
sample = preprocessed_df.select('text', 'clean_text').limit(4).collect()
for i, row in enumerate(sample, 1):
    print(f'Example {i}')
    print('  ORIGINAL:', row['text'][:200])
    print('  CLEAN   :', row['clean_text'][:200])
    print('-' * 80)


Example 1
  ORIGINAL: صور نظام التعليم عند المرأة الأندلسية تستند إلى دراسة دقيقة للمصادر التاريخية التي وثقت حياة العلماء والمثقفين في الأندلس، لا سيما كتب التراجم الشهيرة مثل "الصلة" لابن بشكوال و"صلة الصلة" لابن الزبير 
  CLEAN   : صور نظام تعليم مراه اندلسيه تستند دراسه دقيقه مصادر تاريخيه وثقت حياه علماء مثقف اندلس، سيما كتب تراجم شهيره صله لابن بشكوال صله صله لابن زبير تكمله لابن ابار، وغير اعمال تتابعت لتسجل مراحل تطور بحث ع
--------------------------------------------------------------------------------
Example 2
  ORIGINAL: انهيار دولة الموحدين يعود بشكل كبير للعوامل الثقافية، التي لا تقل أهمية عن العوامل السياسية والعسكرية. المبادئ الفكرية لعقيدة ابن تومرت، التي شكلت الأساس لقيام هذه الدولة، كانت تتضمن تناقضات فكرية وتش
  CLEAN   : انهيار دوله موحد يعود بشكل كبير عوامل ثقافيه، تقل اهميه عوامل سياسيه عسكريه مبادئ فكريه لعقيده ابن تومرت، شكلت اساس لقيام دوله، تتضمن تناقض فكريه وتشريعيه كبيره، جعل عرضه انتقاد عديد فئات اجتماعيه، وم
-----------------------------------------------

## 4. Text length comparison (original vs cleaned)

In [ ]:
length_stats = (
    preprocessed_df
    .withColumn('orig_len',  F.length('text'))
    .withColumn('clean_len', F.length('clean_text'))
    .agg(
        F.avg('orig_len').alias('avg_original_chars'),
        F.avg('clean_len').alias('avg_clean_chars'),
        F.min('clean_len').alias('min_clean_chars'),
        F.max('clean_len').alias('max_clean_chars'),
    )
)
length_stats.show()


+------------------+-----------------+---------------+---------------+
|avg_original_chars|  avg_clean_chars|min_clean_chars|max_clean_chars|
+------------------+-----------------+---------------+---------------+
| 671.7419408679066|475.8533142584645|              0|          13030|
+------------------+-----------------+---------------+---------------+



## 5. Class distribution after preprocessing

In [ ]:
(
    preprocessed_df
    .groupBy('label')
    .count()
    .withColumn('class', F.when(F.col('label') == 0, 'Human').otherwise('AI'))
    .orderBy('label')
    .show()
)


+-----+-----+-----+
|label|count|class|
+-----+-----+-----+
|    0| 8388|Human|
|    1|33552|   AI|
+-----+-----+-----+



## 6. MapReduce-style word count

This demonstrates the Map/Reduce paradigm for corpus statistics
(Phase 2 / Task 2.1, plus Suggested MapReduce Task).

In [ ]:
print('── Top 20 words: full corpus ──')
mapreduce_word_count(preprocessed_df, 'clean_text').show(20, truncate=False)


── Top 20 words: full corpus ──
+--------+-----+
|word    |count|
+--------+-----+
|دراسه   |54620|
|بحث     |42888|
|تحليل   |17068|
|نتائج   |16225|
|تهدف    |13071|
|تاثير   |12352|
|اهميه   |12181|
|فهم     |11652|
|دور     |11393|
|ضوء     |11051|
|اجتماعيه|10285|
|تركيز   |10218|
|جزائر   |9879 |
|استخدام |9870 |
|مجتمع   |9409 |
|لغه     |9203 |
|يهدف    |8702 |
|عربيه   |8660 |
|تعزيز   |8633 |
|جزائري  |8567 |
+--------+-----+
only showing top 20 rows


## 7. Word count by label

In [ ]:
for label_val, label_name in [(0, 'Human'), (1, 'AI')]:
    print(f'── Top 10 words [{label_name}] ──')
    sub = preprocessed_df.filter(F.col('label') == label_val)
    mapreduce_word_count(sub, 'clean_text').show(10, truncate=False)


── Top 10 words [Human] ──
+------+-----+
|word  |count|
+------+-----+
|دراسه |6567 |
|بحث   |2689 |
|مجتمع |1972 |
|اهم   |1923 |
|جزائر |1902 |
|مستوي |1829 |
|لغه   |1811 |
|جزائري|1703 |
|خاصه  |1687 |
|عربيه |1642 |
+------+-----+
only showing top 10 rows
── Top 10 words [AI] ──
+-----+-----+
|word |count|
+-----+-----+
|دراسه|48053|
|بحث  |40199|
|تحليل|16094|
|نتائج|14721|
|تهدف |11854|
|تاثير|11519|
|فهم  |11256|
|اهميه|11051|
|ضوء  |10203|
|دور  |10014|
+-----+-----+
only showing top 10 rows


## 8. Save top-50 word tables

In [ ]:
for label_val, label_name in [(0, 'human'), (1, 'ai')]:
    top = (
        mapreduce_word_count(
            preprocessed_df.filter(F.col('label') == label_val),
            'clean_text'
        )
        .limit(50)
        .toPandas()
    )
    path = FIGURES_DIR / f'top_words_{label_name}.csv'
    top.to_csv(path, index=False, encoding='utf-8-sig')
    print(f'Saved: {path}')


Saved: /content/drive/MyDrive/MSBDA-801-Project/arabic_ai_detection/reports/figures/top_words_human.csv
Saved: /content/drive/MyDrive/MSBDA-801-Project/arabic_ai_detection/reports/figures/top_words_ai.csv


## 9. Bigram frequency analysis

In [ ]:
for label_val, label_name in [(0, 'Human'), (1, 'AI')]:
    print(f'── Top 10 bigrams [{label_name}] ──')
    sub = preprocessed_df.filter(F.col('label') == label_val)
    mapreduce_ngram_frequency(sub, n=2, text_col='clean_text').show(10, truncate=False)


── Top 10 bigrams [Human] ──
+-------------+-----+
|ngram        |count|
+-------------+-----+
|تهدف دراسه   |769  |
|كلم مفتاحيه  |569  |
|ورقه بحثيه   |547  |
|مشرع جزائري  |480  |
|منهج وصفي    |474  |
|لغه عربيه    |471  |
|هدفت دراسه   |461  |
|توصلت دراسه  |306  |
|تسليط ضوء    |298  |
|دلاله احصائيه|297  |
+-------------+-----+
only showing top 10 rows
── Top 10 bigrams [AI] ──
+-------------+-----+
|ngram        |count|
+-------------+-----+
|تهدف دراسه   |8634 |
|يهدف بحث     |6867 |
|ورقه بحثيه   |4587 |
|يتناول بحث   |4091 |
|دراسه تحليل  |3137 |
|بحث دراسه    |3069 |
|تشير نتائج   |3052 |
|نتائج متوقعه |3044 |
|دراسه استكشاف|2981 |
|لغه عربيه    |2893 |
+-------------+-----+
only showing top 10 rows
